# Autocallable: vega by expiry and skew
A phoenix autocallable on TICKER1 (local vol): the vega per surface node by AAD, read by expiry (level, cumulative)
and as a skew sensitivity, then checked against repricings on a bumped vol matrix (same paths): parallel sticky-strike
shifts and skew steepening / flattening.
Docs: *Models and Pricing* (Products: Autocallable; Models: Local volatility).

In [ ]:
import os
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorquant as tq

## Market and product
Latest snapshot in `marketdata/`; 3Y phoenix, semi-annual coupons of 4% with memory (barrier 70%), autocall from 1Y
(100%), put at maturity below 60%. Investor's side, notional 100.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")
E = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
vs = eq.vol_surface
S0 = float(eq.spot)
T_NODES = np.array([float(t) for t in vs.maturity])
K_NODES = np.array([float(k) for k in vs.strike])

cal, mf = tq.TARGET(), tq.BusinessDayConvention.ModifiedFollowing
sg = tq.ScheduleGenerator(cal, mf)
end = cal.advance(E, 3, tq.TimeUnit.Years, mf)
coupon_dates = sg.generate(E, end, 6, tq.TimeUnit.Months)[1:]
autocall_dates = sg.generate(cal.advance(E, 1, tq.TimeUnit.Years, mf), end, 6, tq.TimeUnit.Months)


def autocallable():
    return tq.AutocallableOption(
        ccy=tq.Currency.EUR, notional=100.0, start_date=E, end_date=end, strike=S0,
        coupon_fixing_dates=coupon_dates, coupon_payment_dates=coupon_dates, coupon_rates=[4.0] * len(coupon_dates),
        coupon_barriers=[70.0] * len(coupon_dates), memory=True,
        autocall_fixing_dates=autocall_dates, autocall_payment_dates=autocall_dates,
        autocall_barrier=[100.0] * len(autocall_dates), payoff_barrier=60.0, underlying=TICKER)


print(f"{E}  {TICKER} spot {S0:.2f}  surface {len(T_NODES)} expiries x {len(K_NODES)} strikes")

## Price and vega per node (AAD)
One pricing under the tape; the gradient on `volatility_matrix` is the vega of every node (per vol point: / 100).
Barriers smoothed (1% of the strike, the default) so that the digitals have a pathwise derivative.

In [ ]:
pricer = tq.AutocallableMCPricer(n_paths=20_000, seed=7)
product = autocallable()
pricer.price(product, mkt, autodiff=True)
vol = vs.volatility_matrix
vega = np.asarray(pricer.tape.gradient(product.price, vol)) / 100.0
P0 = float(product.price)
print(f"price {P0:.4f} (% notional)  +/- {float(product.standard_error):.4f}")

nodes = pd.DataFrame(vega, index=[f"{t:.2f}y" for t in T_NODES], columns=[f"{k / S0:.0%}" for k in K_NODES])
nodes.round(4)

Along a row the signs alternate: one node moves the slope and the curvature of the smile, so the local vol is raised at
that strike and lowered next to it. The rows and smooth shapes are the stable reading.

## Vega by expiry and skew
A deformation $\Delta\sigma_{ij}$ of the nodes (vol points) moves the price, to first order, by
$\sum_{ij} \text{vega}_{ij}\,\Delta\sigma_{ij}$. Level of an expiry: $\Delta\sigma_{ij} = 1$ on its row. Skew: the
row rotated around its forward, $\Delta\sigma_{ij} = (1 - m_{ij}) / 10\%$ with $m_{ij} = K_j / F(T_i)$ held within
50%-150% (`tq.skew_weights`): +1 vol point per 10% of strike below the forward, -1 per 10% above.

In [ ]:
W = tq.skew_weights(eq)
by_expiry = tq.vega_by_expiry(vega, eq)
print(pd.DataFrame(W, index=nodes.index, columns=nodes.columns).round(2).to_string(), "\n")
by_expiry.round(4)

## Repricing on a forced vol matrix
`volatility_matrix.assign` replaces the surface (sticky strike: spot, forward and strikes unchanged), the local vol
is rebuilt on it and the product repriced on the same paths (same seed); the matrix is restored afterwards.

In [ ]:
BASE = vol.numpy().copy()


def reprice(shift):
    """Price with the vol nodes shifted by *shift* (vol points, [n_T, n_K])."""
    vol.assign(BASE + shift / 100.0)
    try:
        p = autocallable()
        pricer.price(p, mkt)
        return float(p.price)
    finally:
        vol.assign(BASE)


def check(shape, sizes, aad):
    """Moves +h and -h of *shape*: repriced changes, the AAD prediction, the central slope (first order: it
    should match the AAD) and the second-order part (up + down) / 2."""
    rows = []
    for h in sizes:
        up, dn = reprice(h * shape) - P0, reprice(-h * shape) - P0
        rows.append({"h": h, "repriced +h": up, "repriced -h": dn, "AAD +h": aad * h,
                     "central slope": (up - dn) / (2 * h), "AAD slope": aad, "second order": (up + dn) / 2})
    out = pd.DataFrame(rows).set_index("h")
    out["slope error (%)"] = 100 * (out["central slope"] / out["AAD slope"] - 1)
    return out

### Level: the whole surface up / down (sticky strike)

In [ ]:
ONES = np.ones_like(BASE)
check(ONES, (1.0, 3.0), by_expiry["level"].sum()).round(4)

### Level by expiry
One row up / down 1 vol point: the bucket of the three largest expiries.

In [ ]:
rows = []
for i in np.argsort(-np.abs(by_expiry["level"].to_numpy()))[:3]:
    shape = np.zeros_like(BASE)
    shape[i] = 1.0
    up, dn = reprice(shape), reprice(-shape)
    rows.append({"expiry": f"{T_NODES[i]:.2f}y", "AAD": by_expiry["level"][i], "central bump": (up - dn) / 2,
                 "up": up - P0, "down": P0 - dn})
pd.DataFrame(rows).set_index("expiry").round(4)

### Skew: steepening and flattening of the whole surface
Every expiry rotated by $h$ (+h: steeper, puts up and calls down; -h: flatter). The central slope is the
first-order check; the second-order part is the convexity in the skew (a 2-point move is up to 7 points at 60%).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
i1 = int(np.argmin(np.abs(T_NODES - 1.0)))
for h, style in ((0, "-"), (1, "--"), (-1, ":")):
    ax.plot(K_NODES / S0, 100 * (BASE[i1] + h * W[i1] / 100), style, label=f"skew {h:+d}" if h else "market")
ax.set(xlabel="strike / spot", ylabel="vol (%)", title=f"{T_NODES[i1]:.2f}y smile")
ax.legend()
plt.show()

check(W, (0.5, 1.0, 2.0), by_expiry["skew"].sum()).round(4)

### Skew of a single expiry
The first-order number is exact (the bump converges to it as the move shrinks), but a rotation of one row only bends
the surface in time: in the wings the total variance of that expiry crosses its neighbours', the calendar derivative
of the local vol changes sign and the floors act, so the response is not linear already at a fraction of a point.

In [ ]:
rows = []
for i in np.argsort(-np.abs(by_expiry["skew"].to_numpy()))[:2]:
    shape = np.zeros_like(BASE)
    shape[i] = W[i]
    for h in (1.0, 0.25, 0.05):
        up, dn = reprice(h * shape), reprice(-h * shape)
        rows.append({"expiry": f"{T_NODES[i]:.2f}y", "move": h, "AAD": by_expiry["skew"][i],
                     "central bump": (up - dn) / (2 * h), "up": (up - P0) / h, "down": (P0 - dn) / h})
pd.DataFrame(rows).set_index(["expiry", "move"]).round(4)

## Exercises
1. Express the skew scenario in standard deviations, $(1 - m)$ replaced by $-\ln m / (\sigma_{ATM}\sqrt T)$: how
   does the split by expiry change, and is a single-expiry skew more linear?
2. Move the put barrier from 60% to 80%: which expiries carry the vega now, and does the skew sensitivity change sign?
3. Reprice with Heston SLV (`tq.HestonSLVModel`): do the node vegas still alternate along a row?